# Automatic curation, and export for manual curation (Spyglass pipeline, step 2 of 4)

- **Previously** ([`Pipeline_Spyglass_SpikeSorting.ipynb`](Pipeline_Spyglass_SpikeSorting.ipynb),
  step 1): we sorted the recording and registered the raw sorter output in Spyglass as the first
  curation.
- **In this notebook** (step 2): we curate that raw sort **automatically** — Spyglass's
  `MetricCuration` computes quality metrics (SNR, ISI violations, isolation, noise overlap, ...) and
  applies threshold-based labels, saving the result as a new curation. We then **export** the
  recording and sorting to disk so the same sort can also be curated by hand.
- **Next** ([`Pipeline_Spyglass_ManualCuration.ipynb`](Pipeline_Spyglass_ManualCuration.ipynb),
  step 3): we hand-curate the exported sort in the SpikeInterface GUI (it runs in a different
  environment, so it lives in its own notebook).

> **Environment:** run this notebook with the **`spyglass`** kernel (SpikeInterface 0.99).

See [`README.md`](README.md) for the full four-notebook chain.

## Connect to the database

In [1]:
import json
from pathlib import Path
from pprint import pprint

import datajoint as dj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si

# Load config for database connection info
dj_local_conf_path = "/Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json"
dj.config.load(dj_local_conf_path)

# Spyglass stores some parameters as native Python objects (dicts / lists) in the database;
# this flag lets DataJoint serialize and deserialize those blobs instead of rejecting them.
dj.config["enable_python_native_blobs"] = True

# General Spyglass imports (importing common connects to the database)
import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # requires setuptools<82
[2026-06-17 09:57:01,335][INFO]: DataJoint is configured from /Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json
[2026-06-17 09:57:01,588][INFO]: DataJoint 0.14.9 connected to root@localhost:3306
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## Parameters set manually

These must match the values used in the sorting notebook.

In [16]:
### Parameters set manually ###

# Session: the NWB copy that lives in the database (note the trailing underscore)
nwb_file_name = get_nwb_copy_filename("H3022-210806.nwb")  # -> "H3022-210806_.nwb"

# Which shank (sort group) and which epoch (interval) were sorted
sort_group_id = 0
interval_list_name = "01"  # first wake epoch for this session

# Pipeline parameters (must match the values used in Pipeline_Spyglass_SpikeSorting.ipynb)
preproc_param_name = "default"
sorter = "mountainsort5"
sorter_param_name = "default"

# Shared handoff folder for the manual-curation step (notebooks 2 -> 3 -> 4)
export_root = Path("/Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/curation_exports")

## Recover the raw sort

Find the sorting from step 1 and confirm its raw curation exists.

In [17]:
# Re-derive the sorting produced by Pipeline_Spyglass_SpikeSorting.ipynb from the parameters
# above. Every stage is keyed off the recording, so we first recover its recording_id, then the
# sorting_id, then point at the raw curation (curation_id = 0) that notebook registered.
recording_id = (
    sgs.SpikeSortingRecordingSelection
    & {
        "nwb_file_name": nwb_file_name,
        "sort_group_id": sort_group_id,
        "interval_list_name": interval_list_name,
        "preproc_param_name": preproc_param_name,
    }
).fetch1("recording_id")

sorting_id = (
    sgs.SpikeSortingSelection
    & {
        "recording_id": recording_id,
        "sorter": sorter,
        "sorter_param_name": sorter_param_name,
    }
).fetch1("sorting_id")

curation_key = {"sorting_id": sorting_id, "curation_id": 0}
assert sgs.CurationV1 & curation_key, (
    "No raw curation (curation_id = 0) found for this sorting. "
    "Run Pipeline_Spyglass_SpikeSorting.ipynb first."
)
sgs.CurationV1 & {"sorting_id": sorting_id}

sorting_id,curation_id,parent_curation_id,analysis_file_name name of the file,object_id,merges_applied,description
62224b1f-3d21-498f-bdc3-02c7e398ce77,0,-1,H3022-210806_NYOV44E825.nwb,626eed1e-2797-4cee-8720-aa2ea255a509,0,"raw sort, no curation"
62224b1f-3d21-498f-bdc3-02c7e398ce77,1,0,H3022-210806_XN2FAIZD0N.nwb,692a70c8-24de-48ae-9cb0-f322afe74b71,0,after metric curation
62224b1f-3d21-498f-bdc3-02c7e398ce77,2,0,H3022-210806_JUOVTP1TQH.nwb,e805fead-0c19-442e-aa78-1a3a1c05c838,0,after manual curation (SpikeInterface GUI)
62224b1f-3d21-498f-bdc3-02c7e398ce77,3,-1,H3022-210806_N677TE2AU1.nwb,a25469f0-688f-402c-8228-7e8ec78f1d78,0,"raw sort, no curation"


## 1. Automatic curation with quality metrics

`MetricCuration` computes quality metrics (SNR, ISI violations, isolation, noise overlap, ...) and
applies threshold-based labels to flag low-quality units, saving the result as a new curation that
branches off the raw sort.

The `snr < 7.0` and `isi_violation > 0.005` thresholds below are **illustrative only** — chosen to
exclude a few units from this sort, not as physiological quality criteria. Adjust them for your own
data.

In [18]:
# Make sure the upstream parameter sets MetricCuration depends on exist.
sgs.WaveformParameters.insert_default()
sgs.MetricParameters.insert_default()

# Custom label rules of the form {metric: [operator, threshold, [labels]]}. "snr" and
# "isi_violation" are both computed by the "franklab_default" metric set used below. Valid Spyglass
# labels are: reject, noise, artifact, mua, accept. (These thresholds are illustrative only.)
metric_curation_param_name = "wood_snr_isi"
label_params = {
    "snr": ["<", 7.0, ["noise"]],
    "isi_violation": [">", 0.005, ["mua"]],
}
sgs.MetricCurationParameters.insert1(
    {
        "metric_curation_param_name": metric_curation_param_name,
        "label_params": label_params,
        "merge_params": {},
    },
    skip_duplicates=True,
)

In [19]:
# Select the raw sort + the waveform / metric / label parameter sets, then compute the metrics.
metric_curation_key = {
    "sorting_id": sorting_id,
    "curation_id": 0,
    "waveform_param_name": "default_not_whitened",
    "metric_param_name": "franklab_default",
    "metric_curation_param_name": metric_curation_param_name,
}
selection = sgs.MetricCurationSelection.insert_selection(metric_curation_key)
mc_key = {"metric_curation_id": selection["metric_curation_id"]}
sgs.MetricCuration.populate(mc_key)

[10:15:03][WARNING] Spyglass: This row has already been inserted.


{'success_count': 0, 'error_list': []}

In [20]:
# Pull the computed labels, merge groups, and metrics back out...
labels = sgs.MetricCuration.get_labels(mc_key)
merge_groups = sgs.MetricCuration.get_merge_groups(mc_key)
metrics = sgs.MetricCuration.get_metrics(mc_key)

# get_labels returns one (possibly multi-element) array of labels per unit, so test length.
n_units = len(metrics["snr"])
n_labeled = sum(1 for unit_labels in labels.values() if len(unit_labels) > 0)
print(f"{n_labeled} of {n_units} units received a label")
assert n_labeled > 0, (
    "No unit was labeled -- relax the thresholds in `label_params` above, or MetricCuration "
    "will crash on an all-empty curation_label column."
)
pprint({unit_id: list(unit_labels) for unit_id, unit_labels in labels.items() if len(unit_labels) > 0})

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)


4 of 19 units received a label
{5: ['noise'], 12: ['noise', 'mua'], 17: ['mua'], 18: ['mua']}


/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)


In [21]:
# ...and store them as a new curation (curation_id = 1), branching off the raw sort.
# `insert_curation` always appends a new curation_id, so guard against re-runs stacking duplicates.
metric_description = "after metric curation"
existing_metric = sgs.CurationV1 & {"sorting_id": sorting_id, "description": metric_description}
if existing_metric:
    print(
        f"Metric curation already inserted (curation_id {list(existing_metric.fetch('curation_id'))}); "
        "skipping. Delete it first if you want to re-insert with different parameters."
    )
else:
    sgs.CurationV1.insert_curation(
        sorting_id=sorting_id,
        parent_curation_id=0,
        labels=labels,
        merge_groups=merge_groups,
        metrics=metrics,
        description=metric_description,
    )
sgs.CurationV1 & {"sorting_id": sorting_id}

Metric curation already inserted (curation_id [1]); skipping. Delete it first if you want to re-insert with different parameters.


sorting_id,curation_id,parent_curation_id,analysis_file_name name of the file,object_id,merges_applied,description
62224b1f-3d21-498f-bdc3-02c7e398ce77,0,-1,H3022-210806_NYOV44E825.nwb,626eed1e-2797-4cee-8720-aa2ea255a509,0,"raw sort, no curation"
62224b1f-3d21-498f-bdc3-02c7e398ce77,1,0,H3022-210806_XN2FAIZD0N.nwb,692a70c8-24de-48ae-9cb0-f322afe74b71,0,after metric curation
62224b1f-3d21-498f-bdc3-02c7e398ce77,2,0,H3022-210806_JUOVTP1TQH.nwb,e805fead-0c19-442e-aa78-1a3a1c05c838,0,after manual curation (SpikeInterface GUI)
62224b1f-3d21-498f-bdc3-02c7e398ce77,3,-1,H3022-210806_N677TE2AU1.nwb,a25469f0-688f-402c-8228-7e8ec78f1d78,0,"raw sort, no curation"


## 2. Export the sorting for manual curation

The SpikeInterface GUI needs a `SortingAnalyzer`, which only exists in newer SpikeInterface
(0.101+). This `spyglass` environment ships SpikeInterface 0.99, and the GUI is not installed here
at all — so manual curation must happen in the separate `spikeinterface_gui_env`.

To bridge the two environments we export the **raw** recording and sorting (`curation_id = 0`) to
portable on-disk folders. The next notebook loads them, builds an analyzer, and launches the GUI.
We export the raw sort (not the auto-curated one) so the human sees every unit.

In [22]:
export_dir = export_root / str(sorting_id)
export_dir.mkdir(parents=True, exist_ok=True)

recording = sgs.CurationV1.get_recording(curation_key)  # filtered, referenced spike-band recording
sorting = sgs.CurationV1.get_sorting(curation_key)       # raw mountainsort5 sorting

# Binary recording folder + npz sorting folder are both portable across SpikeInterface versions.
recording.save(folder=export_dir / "recording", format="binary", overwrite=True)
sorting.save(folder=export_dir / "sorting", overwrite=True)

print(f"Exported recording + sorting for sorting_id {sorting_id} to:")
print("   ", export_dir.resolve())

[2026-06-17 10:15:06,689][WARNING]: Skipped checksum for file with hash: b8116206-ce71-028e-7a4f-a0cf70b4edd3, and path: /Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/Spyglass/analysis/H3022-210806/H3022-210806_GL3SRK02VF.nwb
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)
[2026-06-17 10:15:07,505][WARNING]: Skipped checksum for file with hash: b8116206-ce71-028e-7a4f-a0cf70b4edd3, and path: /Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/Spyglass/analys

write_binary_recording with n_jobs = 1 and chunk_size = 30000


/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)


write_binary_recording:   0%|          | 0/1220 [00:00<?, ?it/s]

Exported recording + sorting for sorting_id 62224b1f-3d21-498f-bdc3-02c7e398ce77 to:
    /Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/curation_exports/62224b1f-3d21-498f-bdc3-02c7e398ce77


## Next step

The recording and sorting are now exported to a `<sorting_id>/` subfolder of `export_root` (the
exact path is printed above). To curate by hand, open
[`Pipeline_Spyglass_ManualCuration.ipynb`](Pipeline_Spyglass_ManualCuration.ipynb) (step 3 of 4)
with the **`spikeinterface_gui_env`** kernel and set its `export_dir` to that path. After saving
your curation in the GUI, run
[`Pipeline_Spyglass_CompareCurations.ipynb`](Pipeline_Spyglass_CompareCurations.ipynb) (step 4 of 4)
to ingest and compare the rounds.